In [ ]:
# INGESTA MENSUAL DE RUES A BRONZE
# Selecciona aquí el período. Si repites la ejecución, se reemplaza
# el período existente; no se agregan copias de sus registros.

from pathlib import Path
import sys
import logging

import pandas as pd
import yaml

ANIO_INGESTA = 2026
MES_INGESTA = 11

ruta_actual = Path.cwd().resolve()
candidatos = [ruta_actual, *ruta_actual.parents]
PROJECT_ROOT = next(
    (
        ruta for ruta in candidatos
        if (ruta / "config" / "settings.yaml").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró config/settings.yaml. "
        "Abre el notebook dentro del proyecto MercantiRegistro."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.rues import ingestar_rues_mes

with open(
    PROJECT_ROOT / "config" / "settings.yaml",
    encoding="utf-8",
) as archivo:
    config = yaml.safe_load(archivo)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

resultado = ingestar_rues_mes(
    config=config,
    project_root=PROJECT_ROOT,
    anio=ANIO_INGESTA,
    mes=MES_INGESTA,
)

display(pd.DataFrame([resultado]))

2026-10-27 18:03:05,883 | INFO | RUES 2026-11 | Esperados: 0


,anio,mes,registros,lotes,estado,ruta
0,2026,11,0,0,COMPLETADO,Z:\IngenieriaDeDatos\MercantiRegistro\data\bro...


In [63]:
# VALIDACIÓN DE LA INGESTA MENSUAL
# Ejecuta esta celda después de la ingesta. El año, mes, ruta y
# conteo se obtienen directamente del resultado de esa ejecución.

from datetime import date
import duckdb
import pandas as pd
from pathlib import Path

if "resultado" not in globals():
    raise RuntimeError(
        "Ejecuta primero la celda de ingesta; la validación usa su resultado."
    )

anio = int(resultado["anio"])
mes = int(resultado["mes"])
total_ingesta = int(resultado["registros"])
ruta_mes = Path(resultado["ruta"])

if not 1 <= mes <= 12:
    raise ValueError(f"Mes inválido en el resultado de ingesta: {mes}")

fecha_inicio = date(anio, mes, 1)
fecha_fin = (
    date(anio + 1, 1, 1)
    if mes == 12
    else date(anio, mes + 1, 1)
)
inicio_texto = fecha_inicio.strftime("%Y%m%d")
fin_texto = fecha_fin.strftime("%Y%m%d")

if not ruta_mes.is_dir():
    raise FileNotFoundError(f"No existe la carpeta de Bronze: {ruta_mes}")

archivos = sorted(ruta_mes.glob("*.parquet"))
if not archivos and total_ingesta > 0:
    raise FileNotFoundError(
        f"La ingesta reportó {total_ingesta:,} registros, "
        f"pero no hay archivos Parquet en {ruta_mes}."
    )

if archivos:
    with duckdb.connect() as con:
        resumen = con.execute(
            """
            SELECT
                COUNT(*) AS total_registros,
                COUNT(DISTINCT fecha_matricula) AS fechas_distintas,
                MIN(fecha_matricula) AS fecha_minima,
                MAX(fecha_matricula) AS fecha_maxima,
                COUNT(*) FILTER (
                    WHERE fecha_matricula IS NULL
                ) AS fechas_nulas,
                COUNT(*) FILTER (
                    WHERE fecha_matricula < ?
                       OR fecha_matricula >= ?
                ) AS registros_fuera_periodo
            FROM read_parquet(?)
            """,
            [inicio_texto, fin_texto, str(ruta_mes / "*.parquet")],
        ).df()
else:
    resumen = pd.DataFrame([{
        "total_registros": 0,
        "fechas_distintas": 0,
        "fecha_minima": None,
        "fecha_maxima": None,
        "fechas_nulas": 0,
        "registros_fuera_periodo": 0,
    }])

total_parquet = int(resumen.loc[0, "total_registros"])
assert total_parquet == total_ingesta, (
    f"Conteos diferentes: ingesta={total_ingesta:,}, "
    f"Parquet={total_parquet:,}."
)
assert int(resumen.loc[0, "registros_fuera_periodo"]) == 0, (
    f"Hay registros fuera del período {anio}-{mes:02d}."
)
assert int(resumen.loc[0, "fechas_nulas"]) == 0, (
    "Hay registros sin fecha de matrícula."
)

print(f"Período validado: {anio}-{mes:02d}")
print(f"Archivos Parquet encontrados: {len(archivos)}")
print(f"Registros reportados por ingesta: {total_ingesta:,}")
display(resumen)
print("Validación completada.")

Período validado: 2026-10
Archivos Parquet encontrados: 1
Registros reportados por ingesta: 1,746


,total_registros,fechas_distintas,fecha_minima,fecha_maxima,fechas_nulas,registros_fuera_periodo
0,1746,2,20261001,20261002,0,0


Validación completada.


In [66]:
from pathlib import Path
import duckdb

ruta_bronze = PROJECT_ROOT / config["paths"]["bronze"] / "rues"
anios_a_sumar = (2025, 2026)

# Buscar archivos únicamente en las particiones de esos dos años.
archivos = sorted(
    archivo
    for anio in anios_a_sumar
    for archivo in ruta_bronze.glob(f"anio={anio}/mes=*/*.parquet")
)

if not archivos:
    raise FileNotFoundError(
        f"No se encontraron archivos Parquet para {anios_a_sumar} en {ruta_bronze}"
    )

with duckdb.connect() as con:
    resumen_por_mes = con.execute(
        """
        SELECT
            anio,
            mes,
            COUNT(*) AS registros,
            COUNT(*) FILTER (
                WHERE fecha_matricula IS NULL
            ) AS fechas_nulas,
            MIN(fecha_matricula) AS fecha_minima,
            MAX(fecha_matricula) AS fecha_maxima
        FROM read_parquet(?, hive_partitioning = true)
        GROUP BY anio, mes
        ORDER BY anio, mes
        """,
        [[str(archivo) for archivo in archivos]]
    ).df()

display(resumen_por_mes)

total_2025_2026 = int(resumen_por_mes["registros"].sum())

print(f"Archivos Parquet revisados: {len(archivos)}")
print(f"Total de registros Bronze de 2025 y 2026: {total_2025_2026:,}")

,anio,mes,registros,fechas_nulas,fecha_minima,fecha_maxima
0,2025,01,33227,0,20250102,20250131
1,2025,02,35993,0,20250201,20250228
2,2025,03,35690,0,20250301,20250331
3,2025,04,30685,0,20250401,20250430
4,2025,05,30538,0,20250501,20250531
5,2025,06,26544,0,20250601,20250630
6,2025,07,35226,0,20250701,20250731
7,2025,08,27825,0,20250801,20250831
8,2025,09,30846,0,20250901,20250930
9,2025,10,28961,0,20251001,20251031


Archivos Parquet revisados: 75
Total de registros Bronze de 2025 y 2026: 630,831
